# Exploration & transformation pas à pas — NYC TLC (fhvhv)

Ce notebook applique **exactement les mêmes transformations** que
`src/transform/transform.py`, mais en **pandas** et sur **un seul fichier
échantillon**, afin de visualiser le résultat **après chaque étape**.

> Le pipeline de production reste PySpark ; ce notebook sert à comprendre et
> valider la logique étape par étape.

## 0. Imports et lecture de l'échantillon

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
# Un seul fichier échantillon (VTC à haut volume : Uber, Lyft, ...)
CHEMIN = r"C:\Users\mbare\project\uber-tlc-pipeline\data\raw\fhvhv_tripdata_2024-09.parquet"

df = pd.read_parquet(CHEMIN, engine="fastparquet")
print("Dimensions du fichier brut :", df.shape)
df.head()

Dimensions du fichier brut : (5000, 24)


,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tolls,bcf,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B03404,B03404,2024-09-01 00:00:42,2024-09-01 00:04:11,2024-09-01 00:04:54,2024-09-01 00:12:03,263,75,1.64,429,8.45,0.0,0.23,0.75,0.75,0.0,0.00,6.40,Y,N,N,N,N
1,HV0003,B03404,B03404,2024-09-01 00:26:16,2024-09-01 00:27:03,2024-09-01 00:27:49,2024-09-01 00:48:40,263,249,5.66,1251,45.51,0.0,1.25,4.04,2.75,0.0,0.00,29.99,N,N,N,N,N
2,HV0003,B03404,B03404,2024-09-01 00:07:08,2024-09-01 00:07:22,2024-09-01 00:08:57,2024-09-01 00:18:34,161,170,1.48,577,11.83,0.0,0.32,1.05,2.75,0.0,0.00,7.62,N,N,N,N,N
3,HV0003,B03404,B03404,2024-09-01 00:19:33,2024-09-01 00:21:14,2024-09-01 00:21:48,2024-09-01 00:46:17,233,220,11.52,1469,33.35,0.0,0.92,2.96,2.75,0.0,0.00,29.94,N,N,N,N,N
4,HV0003,B03404,B03404,2024-09-01 00:45:51,2024-09-01 00:49:34,2024-09-01 00:49:34,2024-09-01 01:06:46,220,3,6.29,1032,28.59,0.0,0.79,2.53,0.00,0.0,6.38,18.58,N,N,N,N,N


### Schéma réel (colonnes fhvhv)

In [3]:
df.dtypes

hvfhs_license_num               object
dispatching_base_num            object
originating_base_num            object
request_datetime        datetime64[us]
on_scene_datetime       datetime64[us]
pickup_datetime         datetime64[us]
dropoff_datetime        datetime64[us]
PULocationID                     int32
DOLocationID                     int32
trip_miles                     float64
trip_time                        int64
base_passenger_fare            float64
tolls                          float64
bcf                            float64
sales_tax                      float64
congestion_surcharge           float64
airport_fee                    float64
tips                           float64
driver_pay                     float64
shared_request_flag             object
shared_match_flag               object
access_a_ride_flag              object
wav_request_flag                object
wav_match_flag                  object
dtype: object

## 1. Colonne `plateforme` lisible

Le code `hvfhs_license_num` est traduit en nom de plateforme
(HV0002=Juno, HV0003=Uber, HV0004=Via, HV0005=Lyft ; sinon « Autre »).

In [4]:
PLATEFORMES = {"HV0002": "Juno", "HV0003": "Uber", "HV0004": "Via", "HV0005": "Lyft"}
df["plateforme"] = df["hvfhs_license_num"].map(PLATEFORMES).fillna("Autre")
df["plateforme"].value_counts()

plateforme
Uber    3729
Lyft    1271
Name: count, dtype: int64

## 2. Nettoyage (étape par étape)

À chaque règle on affiche le nombre de lignes **avant → après**.

In [5]:
# 2.1 Suppression des doublons
n = len(df); df = df.drop_duplicates()
print(f"Doublons : {n} -> {len(df)}")

Doublons : 5000 -> 5000


In [6]:
# 2.2 Lignes sans pickup / dropoff
n = len(df); df = df.dropna(subset=["pickup_datetime", "dropoff_datetime"])
print(f"pickup/dropoff non nuls : {n} -> {len(df)}")

pickup/dropoff non nuls : 5000 -> 5000


In [7]:
# 2.3 Le trajet doit se terminer apres avoir commence
n = len(df); df = df[df["dropoff_datetime"] > df["pickup_datetime"]]
print(f"dropoff > pickup : {n} -> {len(df)}")

dropoff > pickup : 5000 -> 5000


In [8]:
# 2.4 Duree du trajet plausible : entre 60 et 18000 secondes (1 min a 5 h)
n = len(df); df = df[(df["trip_time"] >= 60) & (df["trip_time"] <= 18000)]
print(f"trip_time in [60, 18000] : {n} -> {len(df)}")

trip_time in [60, 18000] : 5000 -> 5000


In [9]:
# 2.5 Distance plausible : > 0 et < 200 miles
n = len(df); df = df[(df["trip_miles"] > 0) & (df["trip_miles"] < 200)]
print(f"0 < trip_miles < 200 : {n} -> {len(df)}")

0 < trip_miles < 200 : 5000 -> 5000


In [10]:
# 2.6 Tarif de base positif ou nul
n = len(df); df = df[df["base_passenger_fare"] >= 0]
print(f"base_passenger_fare >= 0 : {n} -> {len(df)}")
print("Lignes conservees apres nettoyage :", len(df))

base_passenger_fare >= 0 : 5000 -> 5000
Lignes conservees apres nettoyage : 5000


## 3. Enrichissement (nouvelles colonnes)

On calcule les colonnes dérivées, puis on affiche un aperçu après chaque groupe.

In [11]:
# 3.1 Duree en minutes
df["duree_minutes"] = (df["trip_time"] / 60).round(2)
df[["trip_time", "duree_minutes"]].head()

,trip_time,duree_minutes
0,429,7.15
1,1251,20.85
2,577,9.62
3,1469,24.48
4,1032,17.20


In [12]:
# 3.2 Temps d'attente (pickup - request), en minutes.
#     null si request_datetime manquant ; valeurs negatives -> null
ecart_min = (df["pickup_datetime"] - df["request_datetime"]).dt.total_seconds() / 60
df["temps_attente_min"] = ecart_min.where(df["request_datetime"].notna() & (ecart_min >= 0)).round(2)
df[["request_datetime", "pickup_datetime", "temps_attente_min"]].head()

,request_datetime,pickup_datetime,temps_attente_min
0,2024-09-01 00:00:42,2024-09-01 00:04:54,4.20
1,2024-09-01 00:26:16,2024-09-01 00:27:49,1.55
2,2024-09-01 00:07:08,2024-09-01 00:08:57,1.82
3,2024-09-01 00:19:33,2024-09-01 00:21:48,2.25
4,2024-09-01 00:45:51,2024-09-01 00:49:34,3.72


In [13]:
# 3.3 Vitesse moyenne (mph), protegee contre la division par zero
df["vitesse_mph"] = np.where(
    df["trip_time"] > 0, df["trip_miles"] / (df["trip_time"] / 3600), np.nan
).round(2)
df[["trip_miles", "trip_time", "vitesse_mph"]].head()

,trip_miles,trip_time,vitesse_mph
0,1.64,429,13.76
1,5.66,1251,16.29
2,1.48,577,9.23
3,11.52,1469,28.23
4,6.29,1032,21.94


In [14]:
# 3.4 Revenu total = base + tolls + congestion + airport_fee + tips
df["revenu_total"] = (
    df["base_passenger_fare"]
    + df["tolls"].fillna(0)
    + df["congestion_surcharge"].fillna(0)
    + df["airport_fee"].fillna(0)
    + df["tips"].fillna(0)
).round(2)
df[["base_passenger_fare", "tolls", "congestion_surcharge", "airport_fee", "tips", "revenu_total"]].head()

,base_passenger_fare,tolls,congestion_surcharge,airport_fee,tips,revenu_total
0,8.45,0.0,0.75,0.0,0.00,9.20
1,45.51,0.0,2.75,0.0,0.00,48.26
2,11.83,0.0,2.75,0.0,0.00,14.58
3,33.35,0.0,2.75,0.0,0.00,36.10
4,28.59,0.0,0.00,0.0,6.38,34.97


In [15]:
# 3.5 Taux de reversement au chauffeur = driver_pay / base_passenger_fare (null si base = 0)
df["taux_reversement"] = np.where(
    df["base_passenger_fare"] != 0, df["driver_pay"] / df["base_passenger_fare"], np.nan
).round(2)
df[["driver_pay", "base_passenger_fare", "taux_reversement"]].head()

,driver_pay,base_passenger_fare,taux_reversement
0,6.40,8.45,0.76
1,29.99,45.51,0.66
2,7.62,11.83,0.64
3,29.94,33.35,0.90
4,18.58,28.59,0.65


In [16]:
# 3.6 Indicateurs booleens
df["a_pourboire"] = df["tips"].fillna(0) > 0
df["est_partagee"] = df["shared_match_flag"] == "Y"
df["est_pmr"] = df["wav_request_flag"] == "Y"
df[["tips", "a_pourboire", "shared_match_flag", "est_partagee", "wav_request_flag", "est_pmr"]].head()

,tips,a_pourboire,shared_match_flag,est_partagee,wav_request_flag,est_pmr
0,0.00,False,N,False,N,False
1,0.00,False,N,False,N,False
2,0.00,False,N,False,N,False
3,0.00,False,N,False,N,False
4,6.38,True,N,False,N,False


In [17]:
# 3.7 Dimensions temporelles (depuis pickup_datetime)
#     jour_semaine suit la convention Spark : 1 = dimanche ... 7 = samedi
df["date_trajet"] = df["pickup_datetime"].dt.date
df["annee"] = df["pickup_datetime"].dt.year
df["mois"] = df["pickup_datetime"].dt.month
df["heure"] = df["pickup_datetime"].dt.hour
df["jour_semaine"] = (df["pickup_datetime"].dt.dayofweek + 1) % 7 + 1
df["nom_jour"] = df["pickup_datetime"].dt.day_name()
df["est_weekend"] = df["jour_semaine"].isin([1, 7])
df[["pickup_datetime", "date_trajet", "annee", "mois", "heure", "jour_semaine", "nom_jour", "est_weekend"]].head()

,pickup_datetime,date_trajet,annee,mois,heure,jour_semaine,nom_jour,est_weekend
0,2024-09-01 00:04:54,2024-09-01,2024,9,0,1,Sunday,True
1,2024-09-01 00:27:49,2024-09-01,2024,9,0,1,Sunday,True
2,2024-09-01 00:08:57,2024-09-01,2024,9,0,1,Sunday,True
3,2024-09-01 00:21:48,2024-09-01,2024,9,0,1,Sunday,True
4,2024-09-01 00:49:34,2024-09-01,2024,9,0,1,Sunday,True


## 4. Table de faits nettoyée (`trajets_nettoyes`)

Sous-ensemble de colonnes conservé pour l'analyse fine (comme dans le module).

In [18]:
COLONNES_FAITS = [
    "plateforme", "PULocationID", "DOLocationID",
    "trip_miles", "duree_minutes", "temps_attente_min", "vitesse_mph",
    "base_passenger_fare", "revenu_total", "tips", "driver_pay", "taux_reversement",
    "a_pourboire", "est_partagee", "est_pmr",
    "date_trajet", "annee", "mois", "heure", "jour_semaine", "nom_jour", "est_weekend",
]
faits = df[COLONNES_FAITS]
print("Table de faits :", faits.shape)
faits.head()

Table de faits : (5000, 22)


,plateforme,PULocationID,DOLocationID,trip_miles,duree_minutes,temps_attente_min,vitesse_mph,base_passenger_fare,revenu_total,tips,driver_pay,taux_reversement,a_pourboire,est_partagee,est_pmr,date_trajet,annee,mois,heure,jour_semaine,nom_jour,est_weekend
0,Uber,263,75,1.64,7.15,4.20,13.76,8.45,9.20,0.00,6.40,0.76,False,False,False,2024-09-01,2024,9,0,1,Sunday,True
1,Uber,263,249,5.66,20.85,1.55,16.29,45.51,48.26,0.00,29.99,0.66,False,False,False,2024-09-01,2024,9,0,1,Sunday,True
2,Uber,161,170,1.48,9.62,1.82,9.23,11.83,14.58,0.00,7.62,0.64,False,False,False,2024-09-01,2024,9,0,1,Sunday,True
3,Uber,233,220,11.52,24.48,2.25,28.23,33.35,36.10,0.00,29.94,0.90,False,False,False,2024-09-01,2024,9,0,1,Sunday,True
4,Uber,220,3,6.29,17.20,3.72,21.94,28.59,34.97,6.38,18.58,0.65,True,False,False,2024-09-01,2024,9,0,1,Sunday,True


## 5. Tables agrégées

Les mêmes agrégations que le module, avec ventilation par **plateforme**
(comparaison Uber vs Lyft).

In [19]:
def kpi(g):
    """Indicateurs cles pour un groupe de trajets (memes definitions que Spark)."""
    return pd.Series({
        "nombre_trajets": len(g),
        "revenu_total": round(g["revenu_total"].sum(), 2),
        "revenu_moyen": round(g["revenu_total"].mean(), 2),
        "pourboire_moyen": round(g["tips"].mean(), 2),
        "taux_reversement_moyen": round(g["taux_reversement"].mean(), 2),
        "distance_moyenne": round(g["trip_miles"].mean(), 2),
        "duree_moyenne_min": round(g["duree_minutes"].mean(), 2),
        "temps_attente_moyen_min": round(g["temps_attente_min"].mean(), 2),
        "vitesse_moyenne_mph": round(g["vitesse_mph"].mean(), 2),
        "part_partagee": round(100 * g["est_partagee"].mean(), 2),
        "part_pmr": round(100 * g["est_pmr"].mean(), 2),
    })

### 5.1 `kpi_globaux` — une seule ligne

In [20]:
kpi_globaux = kpi(df).to_frame().T
kpi_globaux

,nombre_trajets,revenu_total,revenu_moyen,pourboire_moyen,taux_reversement_moyen,distance_moyenne,duree_moyenne_min,temps_attente_moyen_min,vitesse_moyenne_mph,part_partagee,part_pmr
0,5000.0,133898.48,26.78,0.79,0.76,4.94,17.84,5.65,15.01,3.34,0.14


### 5.2 `kpi_par_plateforme` — table centrale de comparaison

In [21]:
kpi_par_plateforme = (
    df.groupby("plateforme").apply(kpi, include_groups=False)
      .sort_values("nombre_trajets", ascending=False)
)
kpi_par_plateforme

,nombre_trajets,revenu_total,revenu_moyen,pourboire_moyen,taux_reversement_moyen,distance_moyenne,duree_moyenne_min,temps_attente_moyen_min,vitesse_moyenne_mph,part_partagee,part_pmr
plateforme,,,,,,,,,,,
Uber,3729.0,100693.11,27.00,0.76,0.76,5.06,17.90,5.47,15.32,4.45,0.11
Lyft,1271.0,33205.37,26.13,0.89,0.76,4.57,17.69,6.15,14.10,0.08,0.24


### 5.3 `trajets_par_heure` (heure x plateforme)

In [22]:
trajets_par_heure = (
    df.groupby(["heure", "plateforme"])
      .agg(
          nombre_trajets=("revenu_total", "size"),
          revenu_moyen=("revenu_total", "mean"),
          duree_moyenne_min=("duree_minutes", "mean"),
          temps_attente_moyen_min=("temps_attente_min", "mean"),
      ).round(2).reset_index()
)
trajets_par_heure

,heure,plateforme,nombre_trajets,revenu_moyen,duree_moyenne_min,temps_attente_moyen_min
0,0,Lyft,1271,26.13,17.69,6.15
1,0,Uber,3729,27.00,17.90,5.47


### 5.4 `trajets_par_jour_semaine` (jour x plateforme)

In [23]:
trajets_par_jour_semaine = (
    df.groupby(["jour_semaine", "nom_jour", "est_weekend", "plateforme"])
      .agg(
          nombre_trajets=("revenu_total", "size"),
          revenu_moyen=("revenu_total", "mean"),
      ).round(2).reset_index().sort_values(["jour_semaine", "plateforme"])
)
trajets_par_jour_semaine

,jour_semaine,nom_jour,est_weekend,plateforme,nombre_trajets,revenu_moyen
0,1,Sunday,True,Lyft,1271,26.13
1,1,Sunday,True,Uber,3729,27.00


### 5.5 `trajets_par_annee_mois` (annee, mois x plateforme)

In [24]:
trajets_par_annee_mois = (
    df.groupby(["annee", "mois", "plateforme"])
      .agg(
          nombre_trajets=("revenu_total", "size"),
          revenu_total=("revenu_total", "sum"),
          revenu_moyen=("revenu_total", "mean"),
          distance_moyenne=("trip_miles", "mean"),
      ).round(2).reset_index()
)
trajets_par_annee_mois

,annee,mois,plateforme,nombre_trajets,revenu_total,revenu_moyen,distance_moyenne
0,2024,9,Lyft,1271,33205.37,26.13,4.57
1,2024,9,Uber,3729,100693.11,27.00,5.06


### 5.6 `revenu_par_zone_depart` (zone de depart x plateforme)

In [25]:
revenu_par_zone_depart = (
    df.groupby([df["PULocationID"].rename("zone_id"), "plateforme"])
      .agg(
          nombre_trajets=("revenu_total", "size"),
          revenu_total=("revenu_total", "sum"),
          revenu_moyen=("revenu_total", "mean"),
          distance_moyenne=("trip_miles", "mean"),
      ).round(2).reset_index().sort_values("nombre_trajets", ascending=False)
)
revenu_par_zone_depart.head(15)

,zone_id,plateforme,nombre_trajets,revenu_total,revenu_moyen,distance_moyenne
129,79,Uber,89,2509.95,28.20,4.71
246,148,Uber,84,2643.52,31.47,5.20
55,37,Uber,84,1879.34,22.37,3.86
131,80,Uber,83,3385.63,40.79,5.15
432,255,Uber,77,2329.25,30.25,4.32
53,36,Uber,58,1772.31,30.56,4.38
94,61,Uber,52,892.84,17.17,3.53
178,112,Uber,51,1301.48,25.52,4.43
434,256,Uber,48,1413.82,29.45,4.21
273,164,Uber,46,1273.87,27.69,3.74


## 6. Bilan

Toutes les étapes de `src/transform/transform.py` ont été reproduites en pandas
sur un échantillon unique. Sur ce fichier, les trajets sont concentrés autour de
minuit le 1er du mois (biais de l'échantillonnage « N premières lignes ») : les
tables horaires et journalières sont donc peu variées ici, mais la logique est
identique à celle exécutée sur l'ensemble des données par Spark.